**TO DO: update this**

In this notebook we'll experiment with different parameters for the RecursiveCharacterTextSplitter splitter from LangChain, and decide which are most suitable for our documents.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

from rag_pipeline.parse_documents import parse_document

In [ ]:
# doc1 contains large sections of text

doc1_path = "/Users/ashapatel/Documents/projects/rag_cc/leaflets_and_guidelines/PILs/motion_sickness_PIL.pdf"

# doc 2 contains smaller sections of text

doc2_path = "/Users/ashapatel/Documents/projects/rag_cc/leaflets_and_guidelines/PILs/antacid_PIL.pdf"

My initial concern from looking at how formats differ between leaflets in my dataset is that the amount of text in each section can vary a lot. For example, the sections in document 1 contain a lot of text, while in document 2 they're much shorter, often a single line of text per section. 

If I split the documents into fixed-size chunks, then some chunks will contain a lot more sections than others. Since sections in theory represent distinct ideas / pieces of information, this means chunks could end up containing too many distinct ideas. If one of many small sections within a chunk contained information relevant to a query, then the embedding for the whole chunk might not represent that information very well and it might not be retrieved.

Let's have a look at the two example documents I chose to demonstrate this issue:

In [ ]:
clean_text_1 = parse_document(doc1_path)

doc_1_page_1_clean_text = clean_text_1[0].page_content
print(doc_1_page_1_clean_text)

In [ ]:
clean_text_2 = parse_document(doc2_path)

doc_2_page_1_clean_text = clean_text_2[0].page_content
print(doc_2_page_1_clean_text)

Let's apply a fixed-size splitter, e.g `RecursiveCharacterTextSplitter`, and have a look at the resulting chunks.

Here's how I'd go about choosing `chunk_size` & `chunk_overlap` for this splitter:

- In order to set `chunk_size`, we need to take into account the maximum sequence length of the embedding model that we'll be using. We can revisit this once we've chosen an embedding model, but for now let's work off of the assumption that 256 is a safe number of tokens for most sentence-transformer models. Assuming ~4 characters per token, this gives a chunk size of ~1000.
    - Note that in our production-ready code, parameters such as `chunk_size` will be set in a config file, not hardcoded.
- It is generally recommended that the overlap between chunks should be set to 10-20% of the chunk size. We'll take 15% of 1000 which is 150. It's good to remember that the purpose of this overlap is to maintain context between one chunk and the next. Without overlap, an idea could get split between two chunks. However, setting the overlap too high wastes storage space in the vector store and wastes compute during retrieval.
- `RecursiveCharacterTextSplitter` also has a `separators` parameter which defaults to `["\n\n", "\n", " ", ""]`. This means it first tries to split on paragraphs, then new lines, then words, then characters.

In [ ]:
doc_1_page_1_chunks = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=150
).split_text(doc_1_page_1_clean_text)

In [ ]:
doc_2_page_1_chunks = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=150
).split_text(doc_2_page_1_clean_text)

In [ ]:
for chunk in doc_1_page_1_chunks:
    print(chunk)
    print("====================")

In [ ]:
for chunk in doc_2_page_1_chunks:
    print(chunk)
    print("====================")

For document 1 we can see that a single section is sometimes split over multiple chunks, whereas for document 2 chunks contain multiple sections.

So, what other splitters would be better suited to these documents?

LangChain splitters also have a `split_documents` attribute. The resulting chunks are `langchain_core.documents.base.Document` objects, just like the outputs of our `parse_document` function. This way we can persist the metadata from pages to chunks.

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
)

for document in clean_text:
    for chunk_index, chunk in enumerate(splitter.split_documents([document])):
        print(chunk_index, chunk.metadata)